# 18. Report tables and figures
**Purpose:** turn the saved results into the final tables (CSV) and figures (PNG) that the report and slides use, in one place, with one colour for each method everywhere. Nothing is recomputed here; every number is read from the files of notebooks 05 to 17 (so run those first).
**Writes:** `results/report/table_*.csv` and `results/report/fig_*.png`.
**Colours:** Okabe-Ito palette (safe for colour-blind readers); each method keeps its colour and line style in every figure.

In [ ]:
from config import *
import numpy as np, pandas as pd
RESULTS = ROOT / "results" / "report"
RESULTS.mkdir(parents=True, exist_ok=True)
O = str(OUT_DIR) + "/"          # results of notebooks 01-10 (run those first)
def read_any(name):
    """Read name.csv.gz or name.csv or name.parquet from OUT_DIR."""
    for ext in (".csv.gz", ".csv", ".parquet"):
        p = OUT_DIR / (name + ext)
        if p.exists():
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    raise FileNotFoundError(name)
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 120})
COL = {"Same as last hour": "#000000", "Model C": "#D55E00", "Model B": "#0072B2", "Model A": "#56B4E9", "Historical typical value": "#009E73",
       "Isolation Forest (on raw readings)": "#CC79A7", "Isolation Forest (on model C error)": "#E69F00", "Fixed percentile": "#999999"}
def fix(m): return m.replace(" (no history)","").replace(" (+ yesterday, last week)","").replace(" (+ previous hour)","").replace("Isolation Forest (raw)","Isolation Forest (on raw readings)")

## Table 1: events by length (test period, usable events)

In [ ]:
t1 = pd.read_csv(O + "table_events_usable.csv"); t1.to_csv(RESULTS / "table_1_events_by_length.csv", index=False); t1

## Table 2: accuracy of the predictors (test period)

In [ ]:
t2 = pd.read_csv(O + "predictor_accuracy.csv"); t2 = t2[t2.split == "test"].drop(columns="split")
t2.to_csv(RESULTS / "table_2_prediction_accuracy_test.csv", index=False); t2

## Table 3: main result, events found at equal alarm rates (95% intervals from resampling buildings)

In [ ]:
ci = pd.read_csv(O + "eval_equal_rate_ci_ext.csv")
t3 = ci[ci["mode"] == "per_building"].pivot_table(index="method", columns="rate_pct", values=["recall_pct", "ci_low", "ci_high"])
t3 = ci[ci["mode"] == "per_building"][["method", "rate_pct", "events", "found", "recall_pct", "ci_low", "ci_high", "median_delay_h", "hour_share_pct"]]
t3.to_csv(RESULTS / "table_3_equal_rate_per_building.csv", index=False)
ci[ci["mode"] == "pooled"].to_csv(RESULTS / "table_3b_equal_rate_pooled.csv", index=False)
t3[t3.rate_pct == 1.0].sort_values("recall_pct", ascending=False)

## Table 4: differences between methods (percentage points, with 95% interval)

In [ ]:
d = pd.read_csv(O + "eval_equal_rate_diff_ext.csv"); d.to_csv(RESULTS / "table_4_differences.csv", index=False); d[(d["mode"] == "per_building") & (d.rate_pct == 1.0)]

## Table 5: results at the frozen cut-offs

In [ ]:
s = pd.read_csv(O + "eval_summary_test.csv"); t5 = s[s.scope == "All labeled buildings"][["method", "events", "events_found", "recall", "recall_low", "recall_high", "median_delay_h", "runs", "run_precision", "false_alarm_pct"]]
t5.to_csv(RESULTS / "table_5_frozen_cutoffs.csv", index=False); t5

## Table 6: events found by event length (1% per building)

In [ ]:
p = pd.read_csv(O + "persistence_recall_by_length.csv"); t6 = p[p["mode"] == "per_building"]; t6.to_csv(RESULTS / "table_6_recall_by_length.csv", index=False); t6.pivot(index="method", columns="length", values="recall_pct")

## Figure 1: events found against alarm rate (per building)

In [ ]:
pb = ci[ci["mode"] == "per_building"]
fig, ax = plt.subplots(figsize=(7, 4.2))
for m, g in pb.groupby("method"):
    g = g.sort_values("rate_pct"); ax.plot(g.rate_pct, g.recall_pct, marker="o", color=COL[m], lw=2.2 if m in ("Same as last hour", "Model C") else 1.3, label=m)
    if m in ("Same as last hour", "Model C"): ax.fill_between(g.rate_pct, g.ci_low, g.ci_high, color=COL[m], alpha=0.12, lw=0)
ax.set_xlabel("Share of hours flagged in each building (%)"); ax.set_ylabel("Labeled events found (%)"); ax.set_xticks([0.5, 1, 2]); ax.set_ylim(0, 60)
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0)); ax.set_title("Events found at equal alarm rates (test period, 587 events)", fontsize=10)
fig.tight_layout(); fig.savefig(RESULTS / "fig_1_events_found_vs_alarm_rate.png", dpi=200); plt.close(fig)

## Figure 2: events found by event length (1% per building)

In [ ]:
order = ["1 h", "2-3 h", "4-12 h", ">12 h"]; meths = ["Same as last hour", "Model C", "Historical typical value", "Isolation Forest (on raw readings)"]
t6 = t6.assign(method=t6.method.map(fix)); fig, ax = plt.subplots(figsize=(7.5, 4)); w = 0.2
for i, m in enumerate(meths):
    g = t6[t6.method == m].set_index("length").reindex(order); x = np.arange(4) + (i - 1.5) * w
    ax.bar(x, g.recall_pct, w, color=COL[m], label=m, yerr=[g.recall_pct - g.ci_low, g.ci_high - g.recall_pct], error_kw=dict(lw=0.8, capsize=2))
ax.set_xticks(range(4)); ax.set_xticklabels([f"{o}\n({int(t6[(t6.method=='Model C')&(t6.length==o)].events.iloc[0])} events)" for o in order])
ax.set_ylabel("Events found (%)"); ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2)); ax.set_ylim(0, 80); ax.set_title("Events found by length of event (1% per building)", fontsize=10)
fig.tight_layout(); fig.savefig(RESULTS / "fig_2_events_found_by_length.png", dpi=200); plt.close(fig)

## Figure 3: which inputs Model C relies on (permutation importance, validation)

In [ ]:
imp = pd.read_csv(ROOT / "results" / "importance" / "permutation_importance_model_C_val.csv"); imp = imp[~imp.group.str.startswith("single:")].sort_values("rmsle_increase_mean")
fig, ax = plt.subplots(figsize=(7, 3.6)); ax.barh(imp.group, imp.rmsle_increase_mean, color="#D55E00")
ax.set_xlabel("Rise in validation error (RMSLE) when scrambled"); ax.set_title("What Model C relies on", fontsize=10)
fig.tight_layout(); fig.savefig(RESULTS / "fig_3_model_C_importance.png", dpi=200); plt.close(fig)

## Copies of existing figures and the CU-BEMS table
The persistence figure (GEPIII) and the CU-BEMS figure are made by notebooks 09 and 12; they are copied here so the report uses one folder.

In [ ]:
import shutil
for src in [OUT_DIR / "persistence_test.png", ROOT / "results" / "cu_bems" / "cubems_persistence_24h.png"]:
    if src.exists(): shutil.copy(src, RESULTS / ("fig_copy_" + src.name))
cb = pd.read_csv(ROOT / "results" / "cu_bems" / "cubems_recall_tables.csv")
t7 = cb[cb.group == "all"] if "all" in set(cb.group) else cb
t7.to_csv(RESULTS / "table_7_cubems_recall.csv", index=False); print(sorted(cb.rule.unique())); t7.head(12)

**Check:** every printed table above should match master sections 10.1 to 10.3 and 20.1 to 20.9. Figure files are written next to the tables.